## Example 1: Basic Local Cluster Setup

In [ ]:
from distributed import Client, LocalCluster
import distributed
import time

print("Example 1: Basic Local Cluster Setup")

try:
    print("✓ Basic distributed cluster setup and management:")
    
    # 1. Package information
    print("  1. Package information:")
    print(f"    Distributed version: {distributed.__version__}")
    
    # List main components
    main_components = [attr for attr in dir(distributed) 
                      if not attr.startswith('_') and attr[0].isupper()]
    print(f"    Main components: {main_components[:10]}")  # Show first 10
    
    # 2. Create a local cluster
    print("  2. Creating local cluster:")
    
    # Create a small local cluster for demonstration
    cluster = LocalCluster(
        n_workers=2,          # Number of worker processes
        threads_per_worker=1, # Threads per worker
        memory_limit='512MB', # Memory limit per worker
        silence_logs=False    # Show logs for demonstration
    )
    
    print(f"    Cluster created: {cluster}")
    print(f"    Cluster address: {cluster.scheduler_address}")
    print(f"    Number of workers: {len(cluster.workers)}")
    
    # 3. Connect client to cluster
    print("  3. Connecting client to cluster:")
    
    client = Client(cluster)
    print(f"    Client connected: {client}")
    print(f"    Client dashboard: {client.dashboard_link}")
    
    # Get cluster status
    cluster_status = client.scheduler_info()
    print(f"    Active workers: {len(cluster_status['workers'])}")
    print(f"    Scheduler address: {cluster_status['address']}")
    
    # 4. Test basic computation
    print("  4. Basic distributed computation:")
    
    def square(x):
        """Simple function to square a number."""
        return x ** 2
    
    def add(x, y):
        """Simple function to add two numbers."""
        return x + y
    
    # Submit single task
    future_result = client.submit(square, 10)
    result = future_result.result()
    print(f"    Single task result: square(10) = {result}")
    
    # Submit multiple tasks
    futures = []
    for i in range(5):
        future = client.submit(square, i)
        futures.append(future)
    
    # Collect results
    results = client.gather(futures)
    print(f"    Multiple task results: {results}")
    
    # 5. Map operation
    print("  5. Map operation across cluster:")
    
    numbers = list(range(1, 11))
    map_results = client.map(square, numbers)
    gathered_results = client.gather(map_results)
    
    print(f"    Map input: {numbers}")
    print(f"    Map results: {gathered_results}")
    
    # 6. Cleanup
    print("  6. Cluster cleanup:")
    
    # Close client and cluster
    client.close()
    cluster.close()
    
    print("    Client and cluster closed")
    print("✓ Basic local cluster setup completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: distributed requires dask installation")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    print("  Note: Cluster operations may require system resources")
    raise AssertionError(
        f"distributed example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Futures and Asynchronous Computing

In [ ]:
from distributed import Client, LocalCluster, as_completed, wait
import time
import random

print("Example 2: Futures and Asynchronous Computing")

try:
    print("✓ Futures and asynchronous computing patterns:")
    
    # 1. Set up cluster with context manager
    print("  1. Setting up cluster for futures demo:")
    
    with LocalCluster(n_workers=2, threads_per_worker=1, silence_logs=True) as cluster:
        with Client(cluster) as client:
            print(f"    Cluster ready with {len(client.scheduler_info()['workers'])} workers")
            
            # 2. Future creation and management
            print("  2. Future creation and management:")
            
            def slow_task(n, delay=0.1):
                """Simulate a slow computation."""
                time.sleep(delay)
                return n * n + random.randint(1, 10)
            
            # Create futures with different completion times
            futures = []
            for i in range(5):
                delay = random.uniform(0.05, 0.15)  # Random delay
                future = client.submit(slow_task, i, delay)
                futures.append(future)
                print(f"      Submitted task {i} with ~{delay:.3f}s delay")
            
            # Check initial status
            print(f"    Created {len(futures)} futures")
            pending_count = sum(1 for f in futures if not f.done())
            print(f"    Pending futures: {pending_count}")
            
            # 3. Wait for completion with different strategies
            print("  3. Different completion strategies:")
            
            # Wait for first future to complete
            first_done = next(as_completed(futures))
            first_result = first_done.result()
            print(f"    First completed result: {first_result}")
            
            # Wait for all remaining futures
            remaining_futures = [f for f in futures if not f.done()]
            if remaining_futures:
                wait(remaining_futures)
                print(f"    All remaining {len(remaining_futures)} futures completed")
            
            # Gather all results
            all_results = client.gather(futures)
            print(f"    All results: {all_results}")
            
            # 4. Future dependencies and chaining
            print("  4. Future dependencies and chaining:")
            
            def process_result(x):
                """Process a result further."""
                return f"processed_{x}"
            
            def combine_results(x, y):
                """Combine two results."""
                return f"{x}+{y}"
            
            # Create dependent futures
            base_future = client.submit(slow_task, 10, 0.05)
            processed_future = client.submit(process_result, base_future)
            
            # Create another future and combine
            another_future = client.submit(slow_task, 20, 0.05)
            combined_future = client.submit(combine_results, processed_future, another_future)
            
            # Wait for final result
            final_result = combined_future.result()
            print(f"    Chained computation result: {final_result}")
            
            # 5. Scatter and gather operations
            print("  5. Scatter and gather operations:")
            
            # Create large data to scatter
            large_data = list(range(1000))
            print(f"    Scattering {len(large_data)} items to cluster")
            
            # Scatter data across workers
            scattered_data = client.scatter(large_data, broadcast=True)
            print(f"    Data scattered: {type(scattered_data)}")
            
            def chunk_sum(data_chunk):
                """Sum a chunk of data."""
                return sum(data_chunk)
            
            # Process scattered data in chunks
            chunk_size = 200
            chunk_futures = []
            for i in range(0, len(large_data), chunk_size):
                chunk = large_data[i:i + chunk_size]
                future = client.submit(chunk_sum, chunk)
                chunk_futures.append(future)
            
            # Gather chunk results
            chunk_results = client.gather(chunk_futures)
            total_sum = sum(chunk_results)
            
            print(f"    Processed {len(chunk_futures)} chunks")
            print(f"    Total sum: {total_sum}")
            print(f"    Verification: {sum(large_data)} (should match)")
            
            # 6. Future status monitoring
            print("  6. Future status monitoring:")
            
            # Create some long-running futures
            monitor_futures = []
            for i in range(3):
                future = client.submit(slow_task, i, 0.1)
                monitor_futures.append(future)
            
            # Monitor progress
            start_time = time.time()
            completed_count = 0
            
            while completed_count < len(monitor_futures):
                current_completed = sum(1 for f in monitor_futures if f.done())
                if current_completed > completed_count:
                    completed_count = current_completed
                    elapsed = time.time() - start_time
                    print(f"      {completed_count}/{len(monitor_futures)} futures completed ({elapsed:.2f}s)")
                time.sleep(0.02)  # Small delay to avoid busy waiting
            
            # Final results
            monitor_results = client.gather(monitor_futures)
            print(f"    Monitoring results: {monitor_results}")
    
    print("✓ Futures and asynchronous computing completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: distributed requires proper installation")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    print("  Note: Async operations may require cluster resources")
    raise AssertionError(
        f"distributed example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: Task Scheduling and Resource Management

In [ ]:
from distributed import Client, LocalCluster, Scheduler, Worker
import time
import psutil
import os

print("Example 3: Task Scheduling and Resource Management")

try:
    print("✓ Task scheduling and resource management:")
    
    # 1. Cluster resource configuration
    print("  1. Cluster resource configuration:")
    
    # Get system information
    cpu_count = os.cpu_count()
    memory_info = psutil.virtual_memory()
    available_memory_gb = memory_info.available / (1024**3)
    
    print(f"    System CPUs: {cpu_count}")
    print(f"    Available memory: {available_memory_gb:.2f} GB")
    
    # Configure cluster with resource limits
    cluster_config = {
        'n_workers': min(2, cpu_count),  # Don't exceed system CPUs
        'threads_per_worker': 1,
        'memory_limit': '256MB',  # Conservative memory limit
        'silence_logs': True
    }
    
    print(f"    Cluster config: {cluster_config}")
    
    with LocalCluster(**cluster_config) as cluster:
        with Client(cluster) as client:
            
            # 2. Worker resource monitoring
            print("  2. Worker resource monitoring:")
            
            scheduler_info = client.scheduler_info()
            workers_info = scheduler_info['workers']
            
            for worker_addr, worker_info in workers_info.items():
                print(f"    Worker {worker_addr}:")
                print(f"      Memory limit: {worker_info.get('memory_limit', 'N/A')}")
                print(f"      Threads: {worker_info.get('nthreads', 'N/A')}")
                print(f"      Status: {worker_info.get('status', 'N/A')}")
            
            # 3. Task priority and resource allocation
            print("  3. Task priority and resource allocation:")
            
            def cpu_intensive_task(n, priority_name):
                """CPU intensive task with different priorities."""
                start_time = time.time()
                result = sum(i**2 for i in range(n))
                end_time = time.time()
                return {
                    'priority': priority_name,
                    'result': result,
                    'duration': end_time - start_time,
                    'worker': os.getpid()
                }
            
            def memory_intensive_task(size_mb, task_name):
                """Memory intensive task."""
                # Create data of specified size
                data_size = size_mb * 1024 * 1024 // 8  # 8 bytes per float
                data = [1.0] * min(data_size, 1000000)  # Cap for safety
                
                # Process the data
                processed = sum(data)
                return {
                    'task': task_name,
                    'data_points': len(data),
                    'result': processed,
                    'worker': os.getpid()
                }
            
            # Submit tasks with different resource requirements
            high_priority_futures = []
            low_priority_futures = []
            
            # High priority (smaller, faster tasks)
            for i in range(3):
                future = client.submit(cpu_intensive_task, 10000, f"high_priority_{i}")
                high_priority_futures.append(future)
            
            # Low priority (larger, slower tasks)  
            for i in range(2):
                future = client.submit(cpu_intensive_task, 50000, f"low_priority_{i}")
                low_priority_futures.append(future)
            
            # Memory intensive tasks
            memory_futures = []
            for i in range(2):
                future = client.submit(memory_intensive_task, 10, f"memory_task_{i}")
                memory_futures.append(future)
            
            # 4. Task execution monitoring
            print("  4. Task execution monitoring:")
            
            # Monitor task completion
            all_futures = high_priority_futures + low_priority_futures + memory_futures
            start_monitoring = time.time()
            
            completed_tasks = []
            while len(completed_tasks) < len(all_futures):
                for future in all_futures:
                    if future.done() and future not in completed_tasks:
                        result = future.result()
                        completed_tasks.append(future)
                        elapsed = time.time() - start_monitoring
                        
                        task_name = result.get('priority', result.get('task', 'unknown'))
                        duration = result.get('duration', 0)
                        worker_pid = result.get('worker', 'N/A')
                        
                        print(f"      Task completed: {task_name}")
                        print(f"        Duration: {duration:.4f}s")
                        print(f"        Worker PID: {worker_pid}")
                        print(f"        Total elapsed: {elapsed:.2f}s")
                
                if len(completed_tasks) < len(all_futures):
                    time.sleep(0.01)  # Small delay
            
            # 5. Resource utilization analysis
            print("  5. Resource utilization analysis:")
            
            # Get worker statistics
            try:
                worker_stats = client.run_on_scheduler(lambda dask_scheduler: {
                    worker_addr: {
                        'tasks_completed': len(worker_info.tasks) if hasattr(worker_info, 'tasks') else 0,
                        'memory_used': getattr(worker_info, 'memory_limit', 0),
                        'status': getattr(worker_info, 'status', 'unknown')
                    }
                    for worker_addr, worker_info in dask_scheduler.workers.items()
                })
                
                print(f"    Worker statistics:")
                total_tasks = 0
                for worker_addr, stats in worker_stats.items():
                    tasks_completed = stats.get('tasks_completed', 0)
                    total_tasks += tasks_completed
                    print(f"      Worker {worker_addr}: {tasks_completed} tasks completed")
                
                print(f"    Total tasks across all workers: {total_tasks}")
                
            except Exception as e:
                print(f"    Worker statistics unavailable: {type(e).__name__}")
                print(f"    Completed {len(all_futures)} tasks across available workers")
                raise
            
            # 6. Load balancing demonstration
            print("  6. Load balancing demonstration:")
            
            def balanced_workload(worker_id, work_amount):
                """Simulate balanced work across workers."""
                import os
                import time
                
                start_time = time.time()
                # Simulate work
                result = sum(i for i in range(work_amount))
                end_time = time.time()
                
                return {
                    'worker_id': worker_id,
                    'actual_pid': os.getpid(),
                    'work_amount': work_amount,
                    'result': result,
                    'duration': end_time - start_time
                }
            
            # Submit balanced workload
            balanced_futures = []
            work_amounts = [1000, 2000, 1500, 1800, 1200]  # Varied work amounts
            
            for i, work_amount in enumerate(work_amounts):
                future = client.submit(balanced_workload, f"task_{i}", work_amount)
                balanced_futures.append(future)
            
            # Collect results
            balanced_results = client.gather(balanced_futures)
            
            # Analyze load distribution
            worker_loads = {}
            for result in balanced_results:
                pid = result['actual_pid']
                if pid not in worker_loads:
                    worker_loads[pid] = {'tasks': 0, 'total_duration': 0, 'total_work': 0}
                
                worker_loads[pid]['tasks'] += 1
                worker_loads[pid]['total_duration'] += result['duration']
                worker_loads[pid]['total_work'] += result['work_amount']
            
            print(f"    Load distribution across workers:")
            for pid, load_info in worker_loads.items():
                avg_duration = load_info['total_duration'] / load_info['tasks']
                print(f"      Worker PID {pid}: {load_info['tasks']} tasks, avg duration: {avg_duration:.4f}s")
            
    print("✓ Task scheduling and resource management completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: distributed requires dask and system monitoring libraries")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    print("  Note: Resource management depends on system capabilities")
    raise AssertionError(
        f"distributed example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: Error Handling and Fault Tolerance

In [ ]:
from distributed import Client, LocalCluster, as_completed
import time
import random

print("Example 4: Error Handling and Fault Tolerance")

try:
    print("✓ Error handling and fault tolerance in distributed computing:")
    
    with LocalCluster(n_workers=2, threads_per_worker=1, silence_logs=True) as cluster:
        with Client(cluster) as client:
            
            # 1. Handling task failures
            print("  1. Handling task failures:")
            
            def unreliable_task(task_id, failure_rate=0.3):
                """Task that "randomly" fails: seeded per task, so the same tasks fail on every run."""
                import random
                import time
                
                time.sleep(0.05)  # Small delay to simulate work
                
                if random.Random(task_id).random() < failure_rate:
                    raise ValueError(f"Task {task_id} failed randomly")
                
                return f"Task {task_id} succeeded"
            
            def reliable_computation(x):
                """Reliable computation that doesn't fail."""
                return x * 2 + 1
            
            # Submit tasks with potential failures
            print("    Submitting tasks with potential failures:")
            unreliable_futures = []
            
            for i in range(8):
                future = client.submit(unreliable_task, i, 0.4)  # 40% failure rate
                unreliable_futures.append(future)
            
            # Handle results and errors: the remote ValueError is re-raised on the client
            expected_failed_ids = {i for i in range(8) if random.Random(i).random() < 0.4}  # {1, 3, 4, 7}
            task_id_of = {future: i for i, future in enumerate(unreliable_futures)}
            successful_results = []
            failed_tasks = []
            
            for future in as_completed(unreliable_futures):
                task_id = task_id_of[future]
                try:
                    result = future.result()
                except ValueError as e:
                    assert task_id in expected_failed_ids, f"task {task_id} was not meant to fail: {e}"
                    assert str(e) == f"Task {task_id} failed randomly", str(e)
                    failed_tasks.append((future, str(e)))
                    print(f"      Failed: {str(e)}")
                else:
                    assert task_id not in expected_failed_ids, f"expected ValueError from task {task_id}"
                    assert result == f"Task {task_id} succeeded", result
                    successful_results.append(result)
                    print(f"      Success: {result}")
            
            assert len(failed_tasks) == len(expected_failed_ids) == 4, (failed_tasks, expected_failed_ids)
            print(f"    Results: {len(successful_results)} succeeded, {len(failed_tasks)} failed")
            
            # 2. Retry mechanisms
            print("  2. Retry mechanisms for failed tasks:")
            
            def task_with_retries(task_id, max_attempts=3):
                """Task that implements retry logic."""
                import time
                
                # Deterministic flakiness: task retry_<n> fails its first (n % 4) attempts, so
                # retry_3 exhausts all three attempts and every other task recovers.
                failures_before_success = int(task_id.rsplit("_", 1)[1]) % 4
                
                for attempt in range(max_attempts):
                    try:
                        time.sleep(0.02)  # Simulate work
                        
                        if attempt < failures_before_success:
                            raise ConnectionError(f"Attempt {attempt + 1} failed for task {task_id}")
                        
                        return f"Task {task_id} succeeded on attempt {attempt + 1}"
                        
                    except ConnectionError as e:
                        if attempt == max_attempts - 1:  # Last attempt
                            raise e
                        time.sleep(0.01 * (attempt + 1))  # Exponential backoff
                        continue
            
            # Test retry mechanism
            retry_futures = []
            for i in range(5):
                future = client.submit(task_with_retries, f"retry_{i}")
                retry_futures.append(future)
            
            retry_results = []
            retry_failures = []
            
            for future in retry_futures:
                try:
                    result = future.result()
                except ConnectionError as e:
                    retry_failures.append(str(e))
                    print(f"    Final failure: {str(e)}")
                else:
                    retry_results.append(result)
                    print(f"    {result}")
            
            assert retry_results == [
                "Task retry_0 succeeded on attempt 1",
                "Task retry_1 succeeded on attempt 2",
                "Task retry_2 succeeded on attempt 3",
                "Task retry_4 succeeded on attempt 1",
            ], retry_results
            assert retry_failures == ["Attempt 3 failed for task retry_3"], retry_failures
            print(f"    Retry results: {len(retry_results)} succeeded, {len(retry_failures)} failed")
            
            # 3. Exception propagation and debugging
            print("  3. Exception propagation and debugging:")
            
            def debug_task(task_id, should_fail=False):
                """Task for debugging exception propagation."""
                import traceback
                import sys
                
                try:
                    if should_fail:
                        # Create a nested exception
                        def nested_function():
                            raise RuntimeError("Deep nested error")
                        
                        def intermediate_function():
                            nested_function()
                        
                        intermediate_function()
                    
                    return f"Debug task {task_id} completed successfully"
                    
                except Exception as e:
                    # Capture full traceback
                    tb_str = traceback.format_exc()
                    raise RuntimeError(f"Debug task {task_id} failed with full traceback:\n{tb_str}")
            
            # Test exception propagation
            debug_futures = [
                client.submit(debug_task, "success", False),
                client.submit(debug_task, "failure", True),
                client.submit(debug_task, "success2", False)
            ]
            
            for i, future in enumerate(debug_futures):
                try:
                    result = future.result()
                except RuntimeError as e:
                    assert i == 1, f"debug task {i} was not meant to fail: {e}"
                    message = str(e)
                    assert message.startswith("Debug task failure failed with full traceback"), message[:200]
                    assert "nested_function" in message and "Deep nested error" in message, message
                    print(f"    Debug task {i} raised {type(e).__name__} carrying the remote traceback")
                    print(f"      first line -> {message.splitlines()[0]}")
                else:
                    assert i != 1, "expected RuntimeError from the failing debug task"
                    print(f"    Debug task {i}: {result}")
            
            # 4. Worker failure simulation and recovery
            print("  4. Worker failure simulation and recovery:")
            
            def worker_info_task():
                """Get information about the worker executing this task."""
                import os
                import threading
                return {
                    'pid': os.getpid(),
                    'thread_id': threading.get_ident(),
                    'worker_name': f"worker_{os.getpid()}"
                }
            
            # Get initial worker information
            worker_info_futures = [client.submit(worker_info_task) for _ in range(4)]
            worker_infos = client.gather(worker_info_futures)
            
            unique_workers = set(info['pid'] for info in worker_infos)
            print(f"    Tasks distributed across {len(unique_workers)} workers: {unique_workers}")
            
            # 5. Graceful degradation
            print("  5. Graceful degradation strategies:")
            
            def adaptive_task(data, fallback_mode=False):
                """Task that adapts based on available resources."""
                import time
                
                if fallback_mode:
                    # Simple, fast computation
                    time.sleep(0.01)
                    return {"result": sum(data), "mode": "fallback", "items": len(data)}
                else:
                    # More complex computation that might fail
                    if len(data) > 1000:  # Simulate resource constraint
                        raise MemoryError("Dataset too large for full processing")
                    
                    time.sleep(0.05)  # Simulate complex processing
                    result = sum(x**2 for x in data)
                    return {"result": result, "mode": "full", "items": len(data)}
            
            # Test graceful degradation
            test_datasets = [
                list(range(100)),    # Small dataset - should succeed
                list(range(1500)),   # Large dataset - should fail and fallback
                list(range(50)),     # Small dataset - should succeed
                list(range(2000))    # Very large dataset - should fail and fallback
            ]
            
            degradation_results = []
            for i, dataset in enumerate(test_datasets):
                try:
                    # Try full processing first
                    future = client.submit(adaptive_task, dataset, fallback_mode=False)
                    result = future.result()
                    
                except MemoryError as e:
                    assert len(dataset) > 1000, f"dataset {i} ({len(dataset)} items) was not meant to fail: {e}"
                    assert str(e) == "Dataset too large for full processing", str(e)
                    print(f"    Dataset {i} failed, trying fallback: {type(e).__name__}")
                    
                    # Fallback to simpler processing
                    try:
                        fallback_future = client.submit(adaptive_task, dataset, fallback_mode=True)
                        fallback_result = fallback_future.result()
                        degradation_results.append(fallback_result)
                        print(f"    Dataset {i} ({len(dataset)} items): {fallback_result['mode']} processing (recovered)")
                    except Exception as fallback_error:
                        print(f"    Dataset {i} fallback also failed: {type(fallback_error).__name__}")
                        raise
                else:
                    assert len(dataset) <= 1000, f"expected MemoryError for the {len(dataset)}-item dataset"
                    degradation_results.append(result)
                    print(f"    Dataset {i} ({len(dataset)} items): {result['mode']} processing")
            
            assert [(r["mode"], r["items"]) for r in degradation_results] == [
                ("full", 100), ("fallback", 1500), ("full", 50), ("fallback", 2000)
            ], degradation_results
            assert degradation_results[0]["result"] == sum(x**2 for x in range(100))
            assert degradation_results[1]["result"] == sum(range(1500))
            
            # 6. Error aggregation and reporting
            print("  6. Error aggregation and reporting:")
            
            error_summary = {
                'successful_tasks': len(successful_results),
                'failed_tasks': len(failed_tasks),
                'retry_successes': len(retry_results),
                'retry_failures': len(retry_failures),
                'graceful_degradation': sum(1 for r in degradation_results if r.get('mode') == 'fallback'),
                'total_tasks': len(successful_results) + len(failed_tasks) + len(retry_results) + len(retry_failures)
            }
            
            print(f"    Error summary:")
            for metric, count in error_summary.items():
                print(f"      {metric}: {count}")
            
            success_rate = (error_summary['successful_tasks'] + error_summary['retry_successes']) / max(error_summary['total_tasks'], 1)
            print(f"    Overall success rate: {success_rate:.2%}")
    
    print("✓ Error handling and fault tolerance completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: distributed requires proper installation for error handling")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    print("  Note: Error handling tests may produce expected failures")
    raise AssertionError(
        f"distributed example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: distributed Package Capabilities and Production Patterns

In [ ]:
import distributed
from distributed import Client, LocalCluster
import time

print("Example 5: distributed Package Capabilities and Production Patterns")

try:
    print("✓ distributed package capabilities and production patterns:")
    
    # 1. Package overview and architecture
    print("  1. Package capabilities overview:")
    print(f"    Distributed version: {distributed.__version__}")
    
    # List main components
    main_components = [attr for attr in dir(distributed) 
                      if not attr.startswith('_') and hasattr(getattr(distributed, attr), '__module__')]
    print(f"    Main components ({len(main_components)}): {main_components[:15]}")
    
    # Core classes and functions
    core_classes = ['Client', 'LocalCluster', 'Scheduler', 'Worker', 'Future']
    available_classes = [cls for cls in core_classes if hasattr(distributed, cls)]
    print(f"    Core classes available: {available_classes}")
    
    # 2. Production cluster configuration patterns
    print("  2. Production cluster configuration patterns:")
    
    production_configs = {
        "development": {
            "n_workers": 2,
            "threads_per_worker": 1,
            "memory_limit": "1GB",
            "dashboard_address": ":8787"
        },
        "testing": {
            "n_workers": 1,
            "threads_per_worker": 2,
            "memory_limit": "512MB",
            "dashboard_address": None
        },
        "production": {
            "n_workers": 4,
            "threads_per_worker": 2,
            "memory_limit": "4GB",
            "dashboard_address": ":8787"
        }
    }
    
    for env, config in production_configs.items():
        print(f"    {env.upper()} config:")
        for key, value in config.items():
            print(f"      {key}: {value}")
    
    # 3. Comprehensive workflow demonstration
    print("  3. Comprehensive distributed workflow:")
    
    with LocalCluster(
        n_workers=2, 
        threads_per_worker=1, 
        memory_limit='512MB',
        silence_logs=True
    ) as cluster:
        with Client(cluster) as client:
            
            # Production-like data processing pipeline
            def extract_data(source_id):
                """Simulate data extraction."""
                import time
                import random
                time.sleep(0.02)  # Simulate I/O
                
                data_size = random.randint(100, 500)
                return {
                    'source_id': source_id,
                    'data': list(range(data_size)),
                    'timestamp': time.time(),
                    'size': data_size
                }
            
            def transform_data(data_batch):
                """Transform extracted data."""
                import time
                time.sleep(0.01)  # Simulate processing
                
                transformed = {
                    'source_id': data_batch['source_id'],
                    'processed_data': [x * 2 + 1 for x in data_batch['data']],
                    'original_size': data_batch['size'],
                    'processed_size': len(data_batch['data']),
                    'transform_time': time.time()
                }
                return transformed
            
            def load_data(processed_batch):
                """Simulate loading processed data."""
                import time
                time.sleep(0.01)  # Simulate storage I/O
                
                return {
                    'source_id': processed_batch['source_id'],
                    'loaded_records': processed_batch['processed_size'],
                    'load_time': time.time(),
                    'status': 'success'
                }
            
            # Execute ETL pipeline
            print("    Executing ETL pipeline:")
            
            # Extract phase
            extract_futures = []
            for source_id in range(5):
                future = client.submit(extract_data, f"source_{source_id}")
                extract_futures.append(future)
            
            extracted_data = client.gather(extract_futures)
            print(f"      Extracted: {len(extracted_data)} data sources")
            
            # Transform phase
            transform_futures = []
            for data_batch in extracted_data:
                future = client.submit(transform_data, data_batch)
                transform_futures.append(future)
            
            transformed_data = client.gather(transform_futures)
            print(f"      Transformed: {len(transformed_data)} data batches")
            
            # Load phase
            load_futures = []
            for processed_batch in transformed_data:
                future = client.submit(load_data, processed_batch)
                load_futures.append(future)
            
            loaded_results = client.gather(load_futures)
            total_records = sum(result['loaded_records'] for result in loaded_results)
            print(f"      Loaded: {total_records} total records from {len(loaded_results)} batches")
            
            # 4. Performance monitoring and metrics
            print("  4. Performance monitoring:")
            
            # Get scheduler and worker statistics
            scheduler_info = client.scheduler_info()
            
            # Scheduler metrics
            print(f"    Scheduler metrics:")
            print(f"      Address: {scheduler_info['address']}")
            print(f"      Workers: {len(scheduler_info['workers'])}")
            print(f"      Services: {list(scheduler_info.get('services', {}).keys())}")
            
            # Worker metrics
            for worker_addr, worker_info in scheduler_info['workers'].items():
                print(f"    Worker {worker_addr}:")
                print(f"      Status: {worker_info.get('status', 'unknown')}")
                print(f"      Memory limit: {worker_info.get('memory_limit', 'N/A')}")
                print(f"      Threads: {worker_info.get('nthreads', 'N/A')}")
            
            # Task execution metrics
            def performance_task(task_size):
                """Task for performance measurement."""
                import time
                start_time = time.time()
                
                # Simulate computational work
                result = sum(i**2 for i in range(task_size))
                
                end_time = time.time()
                return {
                    'task_size': task_size,
                    'result': result,
                    'duration': end_time - start_time,
                    'throughput': task_size / (end_time - start_time)
                }
            
            # Execute performance benchmark
            perf_futures = []
            task_sizes = [1000, 2000, 3000, 5000, 10000]
            
            for size in task_sizes:
                future = client.submit(performance_task, size)
                perf_futures.append(future)
            
            perf_results = client.gather(perf_futures)
            
            print(f"    Performance benchmark results:")
            for result in perf_results:
                print(f"      Task size {result['task_size']}: {result['duration']:.4f}s ({result['throughput']:.0f} ops/s)")
    
    # 5. Best practices and recommendations
    print("  5. Production best practices:")
    
    best_practices = [
        "Use context managers for automatic cluster cleanup",
        "Configure appropriate memory limits per worker",
        "Monitor cluster health through dashboard",
        "Implement proper error handling and retries",
        "Use scatter/gather for large data sets",
        "Profile tasks to identify bottlenecks",
        "Scale workers based on workload characteristics",
        "Use appropriate scheduler plugins for specific needs",
        "Implement logging and monitoring in production",
        "Test cluster configurations in staging environments"
    ]
    
    for i, practice in enumerate(best_practices, 1):
        print(f"      {i:2d}. {practice}")
    
    # 6. Integration patterns
    print("  6. Integration patterns:")
    
    integration_examples = {
        "Web Applications": "Flask/Django with background task processing",
        "Data Pipelines": "ETL workflows with fault tolerance",
        "Machine Learning": "Distributed training and hyperparameter tuning", 
        "Scientific Computing": "Parallel numerical computations",
        "Real-time Analytics": "Stream processing with windowing",
        "Batch Processing": "Large-scale data transformation jobs",
        "Microservices": "Distributed task queues across services",
        "Cloud Deployment": "Kubernetes and Docker container orchestration"
    }
    
    for pattern, description in integration_examples.items():
        print(f"    {pattern}: {description}")
    
    # 7. Package verification
    print("  7. Package verification:")
    
    verification_checks = {
        "client_available": hasattr(distributed, 'Client'),
        "cluster_available": hasattr(distributed, 'LocalCluster'),
        "futures_available": hasattr(distributed, 'Future'),
        "scheduler_available": hasattr(distributed, 'Scheduler'),
        "worker_available": hasattr(distributed, 'Worker'),
        "version_accessible": hasattr(distributed, '__version__'),
        "imports_successful": True  # We've successfully used the package
    }
    
    print(f"    Verification results:")
    for check, passed in verification_checks.items():
        status = "✓ PASS" if passed else "✗ FAIL"
        print(f"      {status}: {check}")
    
    all_passed = all(verification_checks.values())
    print(f"    Overall status: {'✓ ALL CHECKS PASSED' if all_passed else '✗ SOME CHECKS FAILED'}")
    
    # 8. Deployment considerations
    print("  8. Deployment considerations:")
    
    deployment_notes = [
        f"Package version: {distributed.__version__}",
        "Requires Python 3.8+ for optimal performance",
        "Dask dependency provides DataFrame and Array support",
        "Network configuration important for multi-machine clusters",
        "Dashboard provides real-time cluster monitoring",
        "Memory management critical for large-scale processing",
        "Serialization format affects network performance",
        "Security considerations for production deployments"
    ]
    
    for note in deployment_notes:
        print(f"      • {note}")
    
    print("✓ distributed package capabilities and production patterns completed successfully")
    print("  → Ready for scalable distributed computing in production environments")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: Install distributed package: pip install distributed")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    print("  Note: Distributed computing requires adequate system resources")
    raise AssertionError(
        f"distributed example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")